####  **Inicializace prostředí a validace vstupních dat**

Prvním krokem analýzy je import nezbytných programových knihoven a provedení technické inventury datových souborů. Pro manipulaci s daty je využita knihovna pandas, zatímco moduly plotly a scipy.stats zajišťují pokročilou vizualizaci a následné statistické testování (korelace, hypotézy).

Vlastní proces načítání probíhá automatizovaně skrze cyklus, který prochází dedikovanou složku Data. U každého nalezeného souboru ve formátu .csv je provedena kontrola:

    Čitelnosti: Ověření správného kódování (utf-8) a oddělovačů.

    Rozsahu: Zjištění počtu řádků a sloupců pro kontrolu úplnosti datasetu.

    Struktury: Identifikace názvů sloupců pro pozdější párování dat (např. časové řady nebo regiony).

Výsledkem tohoto bloku je přehledný report (tabulka report_df), který slouží jako „diagnostický protokol“. Ten dává analytikovi okamžitou zpětnou vazbu, zda jsou všechny epidemiologické datasety připraveny k dalšímu zpracování, nebo zda některý soubor vykazuje chyby v integritě.

In [1]:
import pandas as pd
import os
import glob
import plotly.graph_objects as go
import plotly.express as px
import scipy.stats as stats

from scipy import stats
from plotly.subplots import make_subplots


# Cesta ke složce s daty
data_folder = "Data"

all_csv_files = [f for f in os.listdir(data_folder) if f.endswith('.csv')]

print(f"--- Zahájení kontroly datových souborů (Celkem nalezeno: {len(all_csv_files)}) ---\n")

summary = []

for file_name in all_csv_files:
    file_path = os.path.join(data_folder, file_name)
    try:
        df = pd.read_csv(file_path, sep=';', encoding='utf-8')
        
        row_count = len(df)
        col_count = len(df.columns)
        first_col = df.columns[0]
        
        summary.append({
            "Soubor": file_name,
            "Stav": "✅ OK",
            "Řádků": row_count,
            "Sloupců": col_count,
            "První sloupec": first_col
        })
    except Exception as e:
        summary.append({
            "Soubor": file_name,
            "Stav": f"❌ CHYBA: {str(e)[:50]}...",
            "Řádků": 0,
            "Sloupců": 0,
            "První sloupec": "-"
        })

# Převedení výsledků do tabulky
report_df = pd.DataFrame(summary)
display(report_df)

print("\n--- Kontrola dokončena ---")

--- Zahájení kontroly datových souborů (Celkem nalezeno: 23) ---



,Soubor,Stav,Řádků,Sloupců,První sloupec
0,Hospitalizace_umrti_chripka(1994-2024).csv,✅ OK,17763,9,rok
1,Hospitalizace_umrti_Covid-19(2020-2025)_CR.csv,✅ OK,2136,124,Datum - ČR
2,Hospitalizace_umrti_Covid-19(2020-2025)_HKK.csv,✅ OK,1816,124,Datum - Královéhradecký kraj
3,Hospitalizace_umrti_Covid-19(2020-2025)_HMP.csv,✅ OK,2066,124,Datum - Hlavní město Praha
4,Hospitalizace_umrti_Covid-19(2020-2025)_JHC.csv,✅ OK,1794,124,Datum - Jihočeský kraj
5,Hospitalizace_umrti_Covid-19(2020-2025)_JMK.csv,✅ OK,2009,124,Datum - Jihomoravský kraj
6,Hospitalizace_umrti_Covid-19(2020-2025)_KVK.csv,✅ OK,1587,124,Datum - Karlovarský kraj
7,Hospitalizace_umrti_Covid-19(2020-2025)_LBK.csv,✅ OK,1660,124,Datum - Liberecký kraj
8,Hospitalizace_umrti_Covid-19(2020-2025)_MSK.csv,✅ OK,2040,124,Datum - Moravskoslezský kraj
9,Hospitalizace_umrti_Covid-19(2020-2025)_OLK.csv,✅ OK,1768,124,Datum - Olomoucký kraj



--- Kontrola dokončena ---


####  **Předzpracování dat, sjednocení regionálních číselníků a agregace**

Tato fáze představuje kritickou část přípravy dat, kde dochází k transformaci surových heterogenních souborů do jednotné struktury vhodné pro komparativní analýzu. Hlavním problémem vstupních dat je rozdílná metodika kódování regionů (zkratky, kódy NUTS, názvy okresů) a různé časové formáty.

**Klíčové operace provedené v tomto bloku:**

    Implementace převodních číselníků: Byly definovány mapovací struktury (okres_to_kraj_full, mapovani_zkratek), které umožňují sjednotit data na úroveň 14 krajů ČR. To je nezbytné, protože data o chřipce jsou často evidována po okresech, zatímco data o COVID-19 po krajích.

    Normalizace časových řad: Datumové údaje jsou převedeny na standardní formát datetime a u celorepublikových dat COVID-19 jsou vypočteny sedmidenní klouzavé průměry (rolling mean), které vyhlazují víkendové výkyvy v hlášení úmrtí a hospitalizací.

    Logika agregace (funkce agreguj_chripku_na_kraje): Tato funkce automatizuje výpočet proočkovanosti v procentech. Sčítá počty vakcinovaných a demografický základ v rámci daného kraje a sezóny, čímž vytváří srovnatelný ukazatel napříč regiony.

    Dávkové zpracování (Merge & Concat): Kód dynamicky načítá jednotlivé soubory pro kraje, doplňuje k nim příslušné meta-informace (NUTS kód, název kraje) a spojuje je do robustních datových rámců (DataFrame).

Výsledkem je 10 vyčištěných a sjednocených datových celků, které jsou připraveny pro statistické výpočty a generování vizualizací bez rizika nekonzistence v regionálním členění.

In [2]:
data_folder = "Data"

# POMOCNÉ FUNKCE A ČÍSELNÍKY

mapovani_zkratek = {
    'HMP': ('CZ010', 'Hlavní město Praha'),
    'STC': ('CZ020', 'Středočeský kraj'),
    'JHC': ('CZ031', 'Jihočeský kraj'),
    'PLK': ('CZ032', 'Plzeňský kraj'),
    'KVK': ('CZ041', 'Karlovarský kraj'),
    'ULK': ('CZ042', 'Ústecký kraj'),
    'LBK': ('CZ051', 'Liberecký kraj'),
    'HKK': ('CZ052', 'Královéhradecký kraj'),
    'PAK': ('CZ053', 'Pardubický kraj'),
    'VYS': ('CZ063', 'Kraj Vysočina'),
    'JMK': ('CZ064', 'Jihomoravský kraj'),
    'OLK': ('CZ071', 'Olomoucký kraj'),
    'ZLK': ('CZ072', 'Zlínský kraj'),
    'MSK': ('CZ080', 'Moravskoslezský kraj')
}

okres_to_kraj_full = {
    'Benešov': 'Středočeský kraj', 'Beroun': 'Středočeský kraj', 'Kladno': 'Středočeský kraj',
    'Kolín': 'Středočeský kraj', 'Kutná Hora': 'Středočeský kraj', 'Mělník': 'Středočeský kraj', 
    'Mladá Boleslav': 'Středočeský kraj', 'Nymburk': 'Středočeský kraj', 'Praha-východ': 'Středočeský kraj', 
    'Praha-západ': 'Středočeský kraj', 'Příbram': 'Středočeský kraj', 'Rakovník': 'Středočeský kraj',
    'České Budějovice': 'Jihočeský kraj', 'Český Krumlov': 'Jihočeský kraj', 'Jindřichův Hradec': 'Jihočeský kraj',
    'Písek': 'Jihočeský kraj', 'Prachatice': 'Jihočeský kraj', 'Strakonice': 'Jihočeský kraj', 
    'Tábor': 'Jihočeský kraj', 'Plzeň-město': 'Plzeňský kraj', 'Plzeň-jih': 'Plzeňský kraj', 
    'Plzeň-sever': 'Plzeňský kraj', 'Domažlice': 'Plzeňský kraj', 'Klatovy': 'Plzeňský kraj',
    'Rokycany': 'Plzeňský kraj', 'Tachov': 'Plzeňský kraj', 'Cheb': 'Karlovarský kraj',
    'Karlovy Vary': 'Karlovarský kraj', 'Sokolov': 'Karlovarský kraj', 'Děčín': 'Ústecký kraj', 
    'Chomutov': 'Ústecký kraj', 'Litoměřice': 'Ústecký kraj', 'Louny': 'Ústecký kraj',
    'Most': 'Ústecký kraj', 'Teplice': 'Ústecký kraj', 'Ústí nad Labem': 'Ústecký kraj',
    'Česká Lípa': 'Liberecký kraj', 'Jablonec nad Nisou': 'Liberecký kraj', 'Liberec': 'Liberecký kraj',
    'Semily': 'Liberecký kraj', 'Hradec Králové': 'Královéhradecký kraj', 'Jičín': 'Královéhradecký kraj',
    'Náchod': 'Královéhradecký kraj', 'Rychnov nad Kněžnou': 'Královéhradecký kraj',
    'Trutnov': 'Královéhradecký kraj', 'Chrudim': 'Pardubický kraj', 'Pardubice': 'Pardubický kraj',
    'Svitavy': 'Pardubický kraj', 'Ústí nad Orlicí': 'Pardubický kraj', 'Havlíčkův Brod': 'Kraj Vysočina', 
    'Jihlava': 'Kraj Vysočina', 'Pelhřimov': 'Kraj Vysočina', 'Třebíč': 'Kraj Vysočina',
    'Žďár nad Sázavou': 'Kraj Vysočina', 'Blansko': 'Jihomoravský kraj', 'Brno-město': 'Jihomoravský kraj', 
    'Brno-venkov': 'Jihomoravský kraj', 'Břeclav': 'Jihomoravský kraj', 'Hodonín': 'Jihomoravský kraj',
    'Vyškov': 'Jihomoravský kraj', 'Znojmo': 'Jihomoravský kraj', 'Jeseník': 'Olomoucký kraj',
    'Olomouc': 'Olomoucký kraj', 'Prostějov': 'Olomoucký kraj', 'Přerov': 'Olomoucký kraj',
    'Šumperk': 'Olomoucký kraj', 'Kroměříž': 'Zlínský kraj', 'Uherské Hradiště': 'Zlínský kraj',
    'Vsetín': 'Zlínský kraj', 'Zlín': 'Zlínský kraj', 'Bruntál': 'Moravskoslezský kraj',
    'Frýdek-Místek': 'Moravskoslezský kraj', 'Karviná': 'Moravskoslezský kraj', 'Nový Jičín': 'Moravskoslezský kraj',
    'Opava': 'Moravskoslezský kraj', 'Ostrava-město': 'Moravskoslezský kraj'
}

kod_kraje_to_nazev = {
    'CZ010': 'Hlavní město Praha', 'CZ020': 'Středočeský kraj', 'CZ031': 'Jihočeský kraj',
    'CZ032': 'Plzeňský kraj', 'CZ041': 'Karlovarský kraj', 'CZ042': 'Ústecký kraj',
    'CZ051': 'Liberecký kraj', 'CZ052': 'Královéhradecký kraj', 'CZ053': 'Pardubický kraj',
    'CZ063': 'Kraj Vysočina', 'CZ064': 'Jihomoravský kraj', 'CZ071': 'Olomoucký kraj',
    'CZ072': 'Zlínský kraj', 'CZ080': 'Moravskoslezský kraj'
}

nazev_to_kod = {v: k for k, v in kod_kraje_to_nazev.items()}

def agreguj_chripku_na_kraje(df):
    df['Kraj_Název'] = df['uzemni_celek'].map(okres_to_kraj_full)
    
    df_clean = df.dropna(subset=['Kraj_Název']).copy()
    
    df_agg = df_clean.groupby(['Kraj_Název', 'sezona']).agg({
        'pocet_vakcinovanych': 'sum',
        'demografie': 'sum'
    }).reset_index()
    
    # 4. DOPLNĚNÍ Kraj_ID
    df_agg.insert(0, 'Kraj_ID', df_agg['Kraj_Název'].map(nazev_to_kod))
    
    # 5. Výpočet proočkovanosti v %
    df_agg['proockovanost_procenta'] = (df_agg['pocet_vakcinovanych'] / df_agg['demografie']) * 100
    
    return df_agg

def preved_detailni_umrti_na_kraje(df):
    df_new = df.copy()
    # Extrakce kódu kraje z kódu okresu
    df_new['Kod_Kraje'] = df_new['okres_bydliste'].str[:5]
    df_new['Kraj_Název'] = df_new['Kod_Kraje'].map(kod_kraje_to_nazev)
    return df_new

def dopln_id_podle_nazvu(df):
    if 'Kraj_ID' not in df.columns:
        df.insert(0, 'Kraj_ID', df['Kraj'].map(nazev_to_kod))
    return df

# SLOUČENÁ KRAJSKÁ DATA COVID-19 ---
kraj_files = glob.glob(os.path.join(data_folder, "Hospitalizace_umrti_Covid-19(2020-2025)_*.csv"))
kraj_files = [f for f in kraj_files if "_CR.csv" not in f]

seznam_df = []

for file in kraj_files:
    temp_df = pd.read_csv(file, sep=';', encoding='utf-8')
    
    temp_df.rename(columns={temp_df.columns[0]: 'Datum'}, inplace=True)
    
    zkratka = os.path.basename(file).split('_')[-1].replace('.csv', '')
    
    nuts_kod, plny_nazev = mapovani_zkratek.get(zkratka, (zkratka, "Neznámý kraj"))
    
    temp_df.insert(0, 'Kraj_Název', plny_nazev)
    temp_df.insert(0, 'Kraj_ID', nuts_kod) 
    
    seznam_df.append(temp_df)

df_covid_umrti_hosp_kraje = pd.concat(seznam_df, ignore_index=True)
df_covid_umrti_hosp_kraje['Datum'] = pd.to_datetime(df_covid_umrti_hosp_kraje['Datum'], dayfirst=True)

# COVID-19: CELOREPUBLIKOVÁ DATA
df_covid_umrti_hosp_cr = pd.read_csv(os.path.join(data_folder, "Hospitalizace_umrti_Covid-19(2020-2025)_CR.csv"), sep=';', encoding='utf-8')
df_covid_umrti_hosp_cr.rename(columns={df_covid_umrti_hosp_cr.columns[0]: 'Datum'}, inplace=True)
df_covid_umrti_hosp_cr['Datum'] = pd.to_datetime(df_covid_umrti_hosp_cr['Datum'], dayfirst=True)
df_covid_umrti_hosp_cr = df_covid_umrti_hosp_cr.sort_values('Datum')
df_covid_umrti_hosp_cr['umrti_7d_prumer'] = df_covid_umrti_hosp_cr['Počet zemřelých'].rolling(window=7, center=True).mean()
df_covid_umrti_hosp_cr['hosp_7d_prumer'] = df_covid_umrti_hosp_cr['Počet nově hospitalizovaných celkem'].rolling(window=7, center=True).mean()

# OČKOVÁNÍ: COVID-19 
df_covid_ockov_kraje = pd.read_csv(os.path.join(data_folder, "Ockovani_kraj_Covid-19(2020-2025).csv"), sep=';', encoding='utf-8')

# CHŘIPKA: HOSPITALIZACE A ÚMRTÍ 
df_chripka_umrti_hosp_kraje = pd.read_csv(os.path.join(data_folder, "Hospitalizace_umrti_chripka(1994-2024).csv"), sep=';', encoding='utf-8')

# CHŘIPKA: ÚMRTÍ 
df_raw_umrti_detail = pd.read_csv(os.path.join(data_folder, "Umrti_chripka(2015-2024).csv"), sep=';', encoding='utf-8')
df_chripka_umrti_kraje = preved_detailni_umrti_na_kraje(df_raw_umrti_detail)

# CHŘIPKA: KRAJSKÉ SROVNÁNÍ ÚMRTÍ 
df_chripka_umrti_abs_kraje = pd.read_csv(os.path.join(data_folder, "Umrti_chripka_kraje_abs(1994-2023).csv"), sep=';', encoding='utf-8')
df_chripka_umrti_norm_kraje = pd.read_csv(os.path.join(data_folder, "Umrti_chripka_kraje_na100tis(1994-2023).csv"), sep=';', encoding='utf-8')
df_chripka_umrti_perc_kraje = pd.read_csv(os.path.join(data_folder, "Umrti_chripka_kraje_procenta(1994-2023).csv"), sep=';', encoding='utf-8')

df_chripka_umrti_abs_kraje = dopln_id_podle_nazvu(df_chripka_umrti_abs_kraje)
df_chripka_umrti_norm_kraje = dopln_id_podle_nazvu(df_chripka_umrti_norm_kraje)
df_chripka_umrti_perc_kraje = dopln_id_podle_nazvu(df_chripka_umrti_perc_kraje)

# OČKOVÁNÍ: CHŘIPKA 
df_raw_65 = pd.read_csv(os.path.join(data_folder, "Ockovani_chripka(2010-2025)_65plus.csv"), sep=';', encoding='utf-8')
df_raw_vsichni = pd.read_csv(os.path.join(data_folder, "Ockovani_chripka(2010-2025)_vsichni.csv"), sep=';', encoding='utf-8')

df_chripka_ockov_65plus_kraje = agreguj_chripku_na_kraje(df_raw_65)
df_chripka_ockov_vsichni_kraje = agreguj_chripku_na_kraje(df_raw_vsichni)

print("✅ Všech 10 datových celků bylo úspěšně načteno a sjednoceno na úroveň krajů.")

✅ Všech 10 datových celků bylo úspěšně načteno a sjednoceno na úroveň krajů.


#### **Finální audit a verifikace datových sad v paměti**

Před zahájením výpočetní fáze a generováním vizualizací je nezbytné provést komplexní revizi všech objektů v operační paměti. Tento blok kódu slouží jako kontrolní mechanismus, který iteruje skrze definovaný slovník datasets a ověřuje existenci a integritu deseti klíčových datových rámců (DataFrames).

**Hlavní funkce tohoto kroku:**

    Kontrola dostupnosti: Pomocí funkce globals().get() kód zjišťuje, zda byly všechny proměnné po předchozích transformacích správně definovány.

    Kvantitativní přehled: Tabulka final_report poskytuje okamžitou informaci o objemu dat (počet řádků a sloupců) pro každou sledovanou kategorii (COVID-19 vs. chřipka, krajská vs. celorepubliková data).

    Časová validace: Speciální pozornost je věnována časovému rozsahu u pandemických dat, kde kód automaticky extrahuje minimální a maximální datum. To potvrzuje, že analýza skutečně pokrývá celé zamýšlené období let 2020 až 2025.

Tento systematický přístup eliminuje riziko chyb typu „Missing Data“ v pozdějších fázích a slouží jako dokumentace stavu datového modelu, na kterém jsou založeny všechny následné interpretace mortality a hospitalizací.

In [3]:
datasets = {
    "df_covid_umrti_hosp_kraje": "COVID-19 - Sloučené kraje (14 souborů)",
    "df_covid_umrti_hosp_cr": "COVID-19 - Celá ČR",
    "df_covid_ockov_kraje": "Očkování COVID-19 - Krajské",
    "df_chripka_umrti_hosp_kraje": "Chřipka - Historické hosp. (1994-2024)",
    "df_chripka_umrti_kraje": "Chřipka - Detailní úmrtí (2015-2024)",
    "df_chripka_umrti_abs_kraje": "Chřipka - Úmrtí kraje (Absolutně)",
    "df_chripka_umrti_norm_kraje": "Chřipka - Úmrtí kraje (na 100 tis.)",
    "df_chripka_umrti_perc_kraje": "Chřipka - Úmrtí kraje (Procenta)",
    "df_chripka_ockov_65plus_kraje": "Očkování chřipka - 65+",
    "df_chripka_ockov_vsichni_kraje": "Očkování chřipka - Všichni"
}

check_results = []

for var_name, description in datasets.items():
    try:
        obj = globals().get(var_name)
        
        if obj is not None:
            check_results.append({
                "Proměnná": var_name,
                "Popis dat": description,
                "Stav": "✅ Načteno",
                "Počet řádků": len(obj),
                "Počet sloupců": len(obj.columns)
            })
        else:
            check_results.append({
                "Proměnná": var_name,
                "Popis dat": description,
                "Stav": "❌ Chybí (nenalezeno)",
                "Počet řádků": 0,
                "Počet sloupců": 0
            })
    except Exception as e:
        check_results.append({
            "Proměnná": var_name,
            "Popis dat": description,
            "Stav": f"⚠️ Chyba: {str(e)[:30]}",
            "Počet řádků": 0,
            "Počet sloupců": 0
        })

final_report = pd.DataFrame(check_results)
print("--- FINÁLNÍ KONTROLA DATOVÝCH SAD V PAMĚTI ---")
display(final_report)

if 'df_covid_umrti_hosp_kraje' in globals():
    min_date = df_covid_umrti_hosp_kraje['Datum'].min().strftime('%d.%m.%Y')
    max_date = df_covid_umrti_hosp_kraje['Datum'].max().strftime('%d.%m.%Y')
    print(f"\n📅 Časový rozsah COVID-19 (kraje): {min_date} až {max_date}")

--- FINÁLNÍ KONTROLA DATOVÝCH SAD V PAMĚTI ---


,Proměnná,Popis dat,Stav,Počet řádků,Počet sloupců
0,df_covid_umrti_hosp_kraje,COVID-19 - Sloučené kraje (14 souborů),✅ Načteno,25402,126
1,df_covid_umrti_hosp_cr,COVID-19 - Celá ČR,✅ Načteno,2136,126
2,df_covid_ockov_kraje,Očkování COVID-19 - Krajské,✅ Načteno,357518,9
3,df_chripka_umrti_hosp_kraje,Chřipka - Historické hosp. (1994-2024),✅ Načteno,17763,9
4,df_chripka_umrti_kraje,Chřipka - Detailní úmrtí (2015-2024),✅ Načteno,3295,11
5,df_chripka_umrti_abs_kraje,Chřipka - Úmrtí kraje (Absolutně),✅ Načteno,103,33
6,df_chripka_umrti_norm_kraje,Chřipka - Úmrtí kraje (na 100 tis.),✅ Načteno,103,33
7,df_chripka_umrti_perc_kraje,Chřipka - Úmrtí kraje (Procenta),✅ Načteno,103,33
8,df_chripka_ockov_65plus_kraje,Očkování chřipka - 65+,✅ Načteno,195,6
9,df_chripka_ockov_vsichni_kraje,Očkování chřipka - Všichni,✅ Načteno,195,6



📅 Časový rozsah COVID-19 (kraje): 11.03.2020 až 13.01.2026


Po úspěšné verifikaci počtu řádků a sloupců následuje fáze obsahové validace. Tento blok kódu systematicky prochází všech deset připravených datových sad a vypisuje reprezentativní vzorek dat (výřez od 50. do 65. řádku). Výběr řádků uprostřed datasetu, nikoliv na začátku, je záměrný, aby bylo možné ověřit stabilitu dat mimo úvodní (často neúplné) záznamy.

In [4]:
# Seznam vašich finálních datasetů pro kontrolu
datasets = {
    "COVID: Hospitalizace a úmrtí (kraje)": df_covid_umrti_hosp_kraje,
    "COVID: Očkování (kraje)": df_covid_ockov_kraje,
    "Chřipka: Hospitalizace a úmrtí (historie)": df_chripka_umrti_hosp_kraje,
    "Chřipka: Úmrtí (detailní)": df_chripka_umrti_kraje,
    "Chřipka: Úmrtí ABS (historie)": df_chripka_umrti_abs_kraje,
    "Chřipka: Úmrtí na 100tis": df_chripka_umrti_norm_kraje,
    "Chřipka: Úmrtí %": df_chripka_umrti_perc_kraje,
    "Chřipka: Očkování 65+": df_chripka_ockov_65plus_kraje,
    "Chřipka: Očkování všichni": df_chripka_ockov_vsichni_kraje,
    "COVID: Celorepubliková data": df_covid_umrti_hosp_cr
}

for nazev, df in datasets.items():
    print(f"\n--- {nazev} ---")
    display(df.iloc[50:65])


--- COVID: Hospitalizace a úmrtí (kraje) ---


,Kraj_ID,Kraj_Název,Datum,Celkový počet nakažených,Počet hospitalizovaných celkem v daném dni,Počet hospitalizovaných na JIP celkem v daném dni,Počet nově hospitalizovaných celkem,Počet nově hospitalizovaných na JIP,Počet zemřelých,Počet antigenních testů,...,Zemřelí ve věku 45-49 let,Zemřelí ve věku 50-54 let,Zemřelí ve věku 55-59 let,Zemřelí ve věku 60-64 let,Zemřelí ve věku 65-69 let,Zemřelí ve věku 70-74 let,Zemřelí ve věku 75-79 let,Zemřelí ve věku 80-84 let,Zemřelí ve věku 85+ let,Zemřelí - věk neznámý
50,CZ052,Královéhradecký kraj,2020-05-12,2.0,1.0,1.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
51,CZ052,Královéhradecký kraj,2020-05-13,NaN,1.0,1.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
52,CZ052,Královéhradecký kraj,2020-05-14,1.0,1.0,1.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
53,CZ052,Královéhradecký kraj,2020-05-15,1.0,1.0,1.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
54,CZ052,Královéhradecký kraj,2020-05-16,NaN,1.0,1.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
55,CZ052,Královéhradecký kraj,2020-05-17,NaN,1.0,1.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
56,CZ052,Královéhradecký kraj,2020-05-18,NaN,1.0,1.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
57,CZ052,Královéhradecký kraj,2020-05-19,1.0,1.0,1.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
58,CZ052,Královéhradecký kraj,2020-05-20,1.0,1.0,1.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
59,CZ052,Královéhradecký kraj,2020-05-21,NaN,1.0,1.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



--- COVID: Očkování (kraje) ---


,id,datum,vakcina,kraj_nuts_kod,kraj_nazev,vekova_skupina,prvnich_davek,druhych_davek,celkem_davek
50,3faa0eec-be57-45ea-a6c7-efbf7984255a,28.12.2020,Comirnaty,CZ064,Jihomoravský kraj,40-44,120,0,120
51,eb04ac58-c65d-4318-95b9-1f1007257c2e,28.12.2020,Comirnaty,CZ064,Jihomoravský kraj,25-29,72,0,72
52,5ab57c3f-05c7-469e-b1a6-fb968a74cb5e,28.12.2020,Comirnaty,CZ010,Hlavní město Praha,60-64,99,0,99
53,b4890fad-e4b2-44d1-a7c7-31664cef3495,28.12.2020,Comirnaty,CZ064,Jihomoravský kraj,18-24,9,0,9
54,5d7e3b14-4001-4dc7-a968-49fb2f6eb72c,28.12.2020,Comirnaty,CZ010,Hlavní město Praha,50-54,144,0,144
55,64d48b7c-13cf-45c3-947a-95c2969273d0,28.12.2020,Comirnaty,CZ064,Jihomoravský kraj,75-79,11,0,11
56,f734116c-0381-4fc8-8c60-7528baee2655,28.12.2020,Comirnaty,CZ064,Jihomoravský kraj,35-39,108,0,108
57,5d06c970-779b-4030-994a-de9a2a2ee7bb,28.12.2020,Comirnaty,CZ010,Hlavní město Praha,18-24,41,0,41
58,2a641e67-c89d-425b-9368-1537d8d21ae2,28.12.2020,Comirnaty,CZ010,Hlavní město Praha,55-59,120,0,120
59,a1a0adf4-ea06-4666-9296-b2e9280152a9,28.12.2020,Comirnaty,CZ010,Hlavní město Praha,70-74,79,0,79



--- Chřipka: Hospitalizace a úmrtí (historie) ---


...

,Kraj_ID,Kraj_Název,sezona,pocet_vakcinovanych,demografie,proockovanost_procenta
50,CZ063,Kraj Vysočina,2015-2016,18766,95262.0,19.699355
51,CZ063,Kraj Vysočina,2016-2017,20379,97958.0,20.803814
52,CZ063,Kraj Vysočina,2017-2018,21328,100357.0,21.252130
53,CZ063,Kraj Vysočina,2018-2019,23356,102513.0,22.783452
54,CZ063,Kraj Vysočina,2019-2020,25015,104291.0,23.985771
55,CZ063,Kraj Vysočina,2020-2021,25703,105748.0,24.305897
56,CZ063,Kraj Vysočina,2021-2022,27702,107024.0,25.883914
57,CZ063,Kraj Vysočina,2022-2023,27271,108999.0,25.019496
58,CZ063,Kraj Vysočina,2023-2024,29060,110988.0,26.183011
59,CZ063,Kraj Vysočina,2024-2025,28310,112286.0,25.212404



--- Chřipka: Očkování všichni ---


,Kraj_ID,Kraj_Název,sezona,pocet_vakcinovanych,demografie,proockovanost_procenta
50,CZ063,Kraj Vysočina,2015-2016,23225,509475.0,4.558614
51,CZ063,Kraj Vysočina,2016-2017,25101,508952.0,4.931899
52,CZ063,Kraj Vysočina,2017-2018,26191,508916.0,5.146429
53,CZ063,Kraj Vysočina,2018-2019,28285,509274.0,5.553985
54,CZ063,Kraj Vysočina,2019-2020,30094,509813.0,5.902949
55,CZ063,Kraj Vysočina,2020-2021,30893,508852.0,6.071117
56,CZ063,Kraj Vysočina,2021-2022,33269,504025.0,6.600665
57,CZ063,Kraj Vysočina,2022-2023,32977,514777.0,6.406075
58,CZ063,Kraj Vysočina,2023-2024,39147,517960.0,7.557920
59,CZ063,Kraj Vysočina,2024-2025,38666,517647.0,7.469569



--- COVID: Celorepubliková data ---


,Datum,Celkový počet nakažených,Počet hospitalizovaných celkem v daném dni,Počet hospitalizovaných na JIP celkem v daném dni,Počet nově hospitalizovaných celkem,Počet nově hospitalizovaných na JIP,Počet zemřelých,Počet antigenních testů,Počet PCR testů,počet testů CELKEM,...,Zemřelí ve věku 55-59 let,Zemřelí ve věku 60-64 let,Zemřelí ve věku 65-69 let,Zemřelí ve věku 70-74 let,Zemřelí ve věku 75-79 let,Zemřelí ve věku 80-84 let,Zemřelí ve věku 85+ let,Zemřelí - věk neznámý,umrti_7d_prumer,hosp_7d_prumer
50,2020-04-30,109.0,273.0,52.0,19.0,1.0,8.0,NaN,NaN,NaN,...,0.0,1.0,0.0,1.0,1.0,3.0,1.0,0.0,4.285714,11.000000
51,2020-05-01,57.0,244.0,46.0,9.0,1.0,8.0,NaN,NaN,NaN,...,0.0,1.0,0.0,0.0,2.0,3.0,2.0,0.0,4.857143,11.571429
52,2020-05-02,18.0,233.0,47.0,11.0,4.0,1.0,NaN,NaN,NaN,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,4.714286,11.428571
53,2020-05-03,26.0,232.0,45.0,6.0,0.0,3.0,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,1.0,0.0,2.0,0.0,4.428571,11.571429
54,2020-05-04,39.0,236.0,47.0,18.0,5.0,5.0,NaN,NaN,NaN,...,0.0,0.0,2.0,0.0,1.0,0.0,2.0,0.0,3.857143,10.428571
55,2020-05-05,77.0,223.0,37.0,12.0,1.0,3.0,NaN,NaN,NaN,...,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,3.428571,10.714286
56,2020-05-06,79.0,209.0,35.0,6.0,1.0,3.0,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,3.714286,10.000000
57,2020-05-07,60.0,191.0,36.0,11.0,6.0,4.0,NaN,NaN,NaN,...,0.0,0.0,1.0,2.0,0.0,0.0,1.0,0.0,3.714286,9.857143
58,2020-05-08,46.0,185.0,40.0,11.0,7.0,5.0,NaN,NaN,NaN,...,0.0,0.0,1.0,1.0,0.0,1.0,2.0,0.0,3.285714,8.285714
59,2020-05-09,18.0,173.0,34.0,6.0,0.0,3.0,NaN,NaN,NaN,...,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,3.285714,7.857143


#### **Agregace ročních trendů a statistická korelace hospitalizací a úmrtí**

V této části dochází k transformaci denních záznamů na roční sumáře, což umožňuje srovnat dlouhodobý vývoj u sezónní chřipky s dynamickým průběhem pandemie COVID-19. Cílem je identifikovat vztah mezi počtem hospitalizací (či celkovým počtem nakažených) a počtem úmrtí v rámci jednotlivých let.

**Klíčové operace provedené v tomto bloku:**

    Časová agregace: U dat COVID-19 je proveden převod datového typu a následné seskupení (groupby) podle kalendářních roků. U chřipky je provedena sumační agregace z krajských dat na celorepublikovou úroveň.

    Výpočet Pearsonova korelačního koeficientu (r): Uvnitř funkce create_combined_chart dochází k automatickému výpočtu lineární závislosti mezi zátěží systému (hospitalizace/nakažení) a mortalitou. Výsledné r a p-hodnota jsou přímo vloženy do grafů jako textové anotace, což poskytuje okamžitý důkaz o síle vztahu mezi těmito proměnnými.

    Pokročilá vizualizace (Kombinované grafy): Pomocí knihovny plotly je definována šablona grafu, která kombinuje sloupcový graf (bar chart) pro zobrazení objemu nemocných/hospitalizovaných a spojnicový graf (line chart) pro zobrazení úmrtí. Tento "duální" přístup umožňuje snadno sledovat, zda křivka úmrtí kopíruje objem hospitalizací, nebo zda dochází k odchylkám (např. vlivem vakcinace či mutací viru).

Výstupem jsou dva interaktivní grafy, které tvoří základ pro kapitoly o roční dynamice onemocnění. Anotace s korelací přímo v grafech zvyšuje metodickou úroveň práce, neboť vizuální trend ihned doplňuje o exaktní statistický údaj.

In [5]:
# PŘÍPRAVA DAT

# COVID-19: Převod data na rok
df_covid_umrti_hosp_cr['Datum'] = pd.to_datetime(df_covid_umrti_hosp_cr['Datum'])

df_covid_rocni = df_covid_umrti_hosp_cr.assign(rok=df_covid_umrti_hosp_cr['Datum'].dt.year).groupby('rok').agg({
    'Celkový počet nakažených': 'sum',
    'Počet zemřelých': 'sum'
}).reset_index()

# Chřipka: Suma ČR
df_chripka_rocni = df_chripka_umrti_hosp_kraje.groupby('rok').agg({
    'pocet_hosp': 'sum',
    'umrti': 'sum'
}).reset_index()


colors = {'hosp': 'royalblue', 'umrti': 'black'}

def create_combined_chart(df, title, hosp_col, umrti_col):
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=df['rok'],
        y=df[hosp_col],
        name='Hospitalizace',
        marker_color=colors['hosp'],
        opacity=0.7
    ))
    
    fig.add_trace(go.Scatter(
        x=df['rok'],
        y=df[umrti_col],
        name='Úmrtí',
        mode='lines+markers', 
        line=dict(color=colors['umrti'], width=3),
        marker=dict(size=8)
    ))
    
    fig.update_layout(
        title=title,
        xaxis_title='Rok',
        yaxis_title='Počet osob (kumulativně za rok)',
        barmode='group',
        template='plotly_white',
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        hovermode="x unified"
    )
    return fig

def create_combined_chart(df, title, hosp_col, umrti_col):
    clean_df = df[[hosp_col, umrti_col]].dropna()
    r_val, p_val = stats.pearsonr(clean_df[hosp_col], clean_df[umrti_col])
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=df['rok'],
        y=df[hosp_col],
        name='Hospitalizace',
        marker_color=colors['hosp'],
        opacity=0.7
    ))
    
    fig.add_trace(go.Scatter(
        x=df['rok'],
        y=df[umrti_col],
        name='Úmrtí',
        mode='lines+markers', 
        line=dict(color=colors['umrti'], width=3),
        marker=dict(size=8)
    ))
    
    fig.add_annotation(
        xref="paper", yref="paper",
        x=0.02, y=0.95,
        text=f"r = {r_val:.2f}<br>p-hodnota {'< 0.001' if p_val < 0.001 else f'= {p_val:.3f}'}",
        showarrow=False,
        font=dict(size=14, color="black"),
        bgcolor="white",
        bordercolor="black",
        borderwidth=1,
        align="left"
    )
    
    fig.update_layout(
        title=title,
        xaxis_title='Rok',
        yaxis_title='Počet osob (kumulativně za rok)',
        template='plotly_white',
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        hovermode="x unified"
    )
    return fig

fig_covid = create_combined_chart(df_covid_rocni, 'COVID-19: Roční přehled hospitalizací a úmrtí', 'Celkový počet nakažených', 'Počet zemřelých')
fig_covid.show()

fig_chripka = create_combined_chart(df_chripka_rocni, 'Chřipka: Roční přehled hospitalizací a úmrtí (Historie)', 'pocet_hosp', 'umrti')
fig_chripka.show()

#### **Analýza diagnostické aktivity a jejího vlivu na interpretaci dat**

Korektní interpretace epidemiologických dat vyžaduje znalost rozsahu testování v čase. Tento blok kódu sumarizuje roční intenzitu diagnostiky COVID-19 v České republice, přičemž rozlišuje mezi vysoce citlivými metodami PCR a rychlými antigenními testy.

**Hlavní aspekty provedené analýzy:**

    Kumulativní agregace: Data jsou seskupena podle let, což umožňuje přímo porovnat diagnostický tlak v jednotlivých fázích pandemie.

    Skládaný sloupcový graf (Stacked Bar Chart): Vizualizace umožňuje sledovat nejen celkový objem testování, ale i změnu v „diagnostickém mixu“. Zatímco v prvním roce pandemie dominovaly PCR testy, v pozdějších letech (zejména po zavedení plošného testování) výrazně narostl podíl antigenních testů.

    Kontextualizace výsledků: Tento graf slouží jako validační nástroj pro kapitoly o úmrtnosti. Pokud v grafech úmrtnosti pozorujeme pokles, díky tomuto grafu můžeme posoudit, zda tento pokles odpovídá i poklesu diagnostické kapacity, nebo zda jde o nezávislý epidemiologický trend.

Tato data jsou nezbytná pro pochopení věrohodnosti epidemiologických dat – zejména pro vyloučení zkreslení způsobeného nízkým počtem provedených testů v období mimo hlavní pandemické vlny.

In [6]:
# Příprava dat pro testy
df_testy_rocni = df_covid_umrti_hosp_cr.assign(
    rok=pd.to_datetime(df_covid_umrti_hosp_cr['Datum']).dt.year
).groupby('rok').agg({
    'Počet antigenních testů': 'sum',
    'Počet PCR testů': 'sum'
}).reset_index()

fig_testy = go.Figure()
fig_testy.add_trace(go.Bar(x=df_testy_rocni['rok'], y=df_testy_rocni['Počet PCR testů'], name='PCR testy', marker_color='darkblue'))
fig_testy.add_trace(go.Bar(x=df_testy_rocni['rok'], y=df_testy_rocni['Počet antigenních testů'], name='Antigenní testy', marker_color='lightblue'))

fig_testy.update_layout(
    title='Roční přehled diagnostiky COVID-19 v ČR',
    xaxis_title='Rok',
    yaxis_title='Počet provedených testů',
    barmode='stack',
    template='plotly_white',
    hovermode="x unified"
)
fig_testy.show()

#### **Demografická analýza mortality: Sjednocení věkových struktur**

Pro korektní komparaci dopadů COVID-19 a sezónní chřipky je nezbytné sjednotit věkové kategorie obou onemocnění. Tento blok kódu provádí transformaci (re-agregaci) detailních pětiletých věkových skupin COVID-19 do širších standardizovaných intervalů používaných při sledování respiračních nákaz v ČR (0–19, 20–44, 45–64, 65–74, 75–84 a 85+).

**Hlavní kroky zpracování:**

    Mapování a re-binning: Pomocí slovníku mapping_covid dochází k logickému sloučení úzkých věkových skupin do šesti hlavních kategorií. Tento krok zajišťuje, že úmrtnost seniorů (např. 75–79 a 80–84 let) je posuzována jako jeden srovnatelný celek.

    Iterativní agregace: Kód dynamicky prochází sloupce v datasetu a sčítá úmrtí napříč definovanými věkovými intervaly pro každý kalendářní rok.

    Vizualizace věkové pyramidy v čase: Výsledný sloupcový graf (px.bar) využívá seskupený režim (barmode='group'), který umožňuje sledovat dynamiku úmrtnosti v čase pro každou věkovou skupinu zvlášť. Použitá barevná škála (odstíny červené) vizuálně akcentuje rizikovost vyšších věkových kategorií.

Tato vizualizace poskytuje jasný důkaz o věkovém gradientu úmrtnosti. Umožňuje v diplomové práci kvantifikovat, jakou měrou se na celkové mortalitě podílely nejstarší kohorty a jak se tento poměr měnil v jednotlivých vlnách pandemie.

In [7]:
# Sjednocení sloupců COVIDu do věkových skupin chřipky
mapping_covid = {
    '0–19': ['Zemřelí ve věku 0-4 let', 'Zemřelí ve věku 5-9 let', 'Zemřelí ve věku 10-14 let', 'Zemřelí ve věku 15-19 let'],
    '20–44': ['Zemřelí ve věku 20-24 let', 'Zemřelí ve věku 25-29 let', 'Zemřelí ve věku 30-34 let', 'Zemřelí ve věku 35-39 let', 'Zemřelí ve věku 40-44 let'],
    '45–64': ['Zemřelí ve věku 45-49 let', 'Zemřelí ve věku 50-54 let', 'Zemřelí ve věku 55-59 let', 'Zemřelí ve věku 60-64 let'],
    '65–74': ['Zemřelí ve věku 65-69 let', 'Zemřelí ve věku 70-74 let'],
    '75–84': ['Zemřelí ve věku 75-79 let', 'Zemřelí ve věku 80-84 let'],
    '85+': ['Zemřelí ve věku 85+ let']
}

df_covid_prep = df_covid_umrti_hosp_cr.copy()
df_covid_prep['rok'] = pd.to_datetime(df_covid_prep['Datum']).dt.year

covid_age_data = []
for skupina, sloupce in mapping_covid.items():
    existujici = [c for c in sloupce if c in df_covid_prep.columns]
    temp = df_covid_prep.groupby('rok')[existujici].sum().sum(axis=1).reset_index()
    temp.columns = ['rok', 'pocet_umrti']
    temp['vekova_skupina'] = skupina
    covid_age_data.append(temp)

df_covid_age_total = pd.concat(covid_age_data)

fig_covid_age = px.bar(
    df_covid_age_total, 
    x='rok', 
    y='pocet_umrti', 
    color='vekova_skupina',
    title='COVID-19: Vývoj úmrtnosti dle věkových skupin (2020–2026)',
    labels={'pocet_umrti': 'Počet úmrtí', 'rok': 'Rok', 'vekova_skupina': 'Věk'},
    barmode='group',
    category_orders={"vekova_skupina": ['0–19', '20–44', '45–64', '65–74', '75–84', '85+']},
    color_discrete_sequence=px.colors.sequential.Reds_r,
    template='plotly_white'
)
fig_covid_age.update_layout(hovermode="x unified",)
fig_covid_age.show()

#### **Demografická analýza mortality: Historický vývoj úmrtnosti na chřipku**

Tento krok dokončuje proces sjednocení metodiky pro srovnání obou onemocnění. Zaměřuje se na transformaci historických dat o úmrtnosti na chřipku tak, aby jejich struktura odpovídala moderním datům o COVID-19 a umožnila přímou vizuální komparaci věkového rozložení obětí.

**Klíčové operace v tomto bloku:**

    Rekonstrukce datové struktury (Melt): Pomocí funkce melt dochází k transformaci tabulky z širokého formátu na dlouhý (long format). Roky, které byly původně v záhlaví sloupců, jsou převedeny do jediného sloupce rok, což je nezbytný krok pro následné seskupování a vizualizaci.

    Čištění a filtrace: Z analýzy je odstraněna agregovaná kategorie „Celá populace“, aby nedocházelo k duplicitnímu započítávání hodnot a v grafu zůstaly pouze diskrétní věkové kohorty.

    Srovnatelná vizualizace: Graf využívá identické věkové kategorie jako v případě COVID-19. Pro odlišení od pandemických dat je zde zvolena modrá barevná škála (Blues_r). Modrá barva v epidemiologii často symbolizuje „chladnější“ sezónní průběh, zatímco červená (použitá u COVID-19) akcentuje krizový stav.

Díky tomuto postupu lze v diplomové práci vedle sebe položit oba grafy (červený pro COVID-19 a modrý pro chřipku) a okamžitě interpretovat rozdíly v intenzitě zasažení jednotlivých věkových skupin. Jasně se zde ukazuje, zda chřipka vykazuje stejnou míru „přetížení“ nejstarších ročníků jako COVID-19.

In [8]:
roky_chripka = [str(r) for r in range(1994, 2024)]

df_chripka_age = df_chripka_umrti_abs_kraje.melt(
    id_vars=['Věková kategorie'], 
    value_vars=roky_chripka, 
    var_name='rok', 
    value_name='pocet_umrti'
)

df_chripka_age_sum = df_chripka_age.groupby(['rok', 'Věková kategorie'])['pocet_umrti'].sum().reset_index()

df_chripka_age_sum = df_chripka_age_sum[df_chripka_age_sum['Věková kategorie'] != 'Celá populace']
df_chripka_age_sum['rok'] = df_chripka_age_sum['rok'].astype(int)

fig_flu_age = px.bar(
    df_chripka_age_sum, 
    x='rok', 
    y='pocet_umrti', 
    color='Věková kategorie',
    title='Chřipka: Vývoj úmrtnosti dle věkových skupin v ČR (2015–2023)',
    labels={'pocet_umrti': 'Počet úmrtí', 'rok': 'Rok', 'Věková kategorie': 'Věk'},
    barmode='group',
    category_orders={"Věková kategorie": ['0–19', '20–44', '45–64', '65–74', '75–84', '85+']},
    color_discrete_sequence=px.colors.sequential.Blues_r,
    template='plotly_white',
)
fig_flu_age.update_layout(hovermode="x unified",)
fig_flu_age.show()

#### **Srovnání relativní struktury mortality: Analýza věkové distribuce**

V této fázi analýzy přecházíme od absolutních počtů k vyjádření procentuálního podílu jednotlivých věkových skupin na celkové mortalitě. Tento metodický přístup je klíčový pro eliminaci vlivu různé intenzity epidemií v jednotlivých letech a umožňuje identifikovat, které věkové kohorty jsou v rámci daného onemocnění nejvíce ohroženy.

**Klíčové parametry analýzy:**

    Normalizace na 100 % (barnorm='percent'): Tato transformace přepočítává výšku sloupců tak, aby každý rok představoval plných 100 %. Díky tomu lze okamžitě odečíst, jakou část obětí tvořili například senioři nad 85 let ve srovnání s mladšími ročníky.

    Identifikace demografických trendů: U COVID-19 lze tímto způsobem sledovat, zda se s postupem času (nástup vakcinace, proměna variant viru) měnil „věkový profil“ obětí. U chřipky graf odhaluje stabilitu (nebo proměnlivost) věkového složení zemřelých v horizontu téměř tří desetiletí.

    Vizuální komparace (Červená vs. Modrá): Zachování barevné logiky (červené odstíny pro COVID-19, modré pro chřipku) usnadňuje čtenáři orientaci při rychlém listování analytickou částí práce.

Tento typ vizualizace je v diplomové práci zásadním argumentem pro kapitoly o rizikových skupinách. Umožňuje exaktně doložit, zda COVID-19 zasahoval mladší věkové skupiny relativně častěji než sezónní chřipka, nebo zda je profil úmrtnosti u obou onemocnění u seniorů nad 65 let dominantně podobný.

In [9]:
#RELATIVNÍ PODÍL - COVID-19 ---
fig_covid_share = px.bar(
    df_covid_age_total, 
    x='rok', 
    y='pocet_umrti', 
    color='vekova_skupina',
    title='COVID-19: Relativní podíl věkových skupin na celkové úmrtnosti',
    labels={'pocet_umrti': 'Podíl úmrtí', 'rok': 'Rok', 'vekova_skupina': 'Věková skupina'},
    category_orders={"vekova_skupina": ['0–19', '20–44', '45–64', '65–74', '75–84', '85+']},
    color_discrete_sequence=px.colors.sequential.Reds_r,
    template='plotly_white'
)

fig_covid_share.update_layout(barmode='relative', barnorm='percent')
fig_covid_share.show()


# RELATIVNÍ PODÍL - CHŘIPKA ---
fig_flu_share = px.bar(
    df_chripka_age_sum, 
    x='rok', 
    y='pocet_umrti', 
    color='Věková kategorie',
    title='Chřipka: Relativní podíl věkových skupin na celkové úmrtnosti',
    labels={'pocet_umrti': 'Podíl úmrtí', 'rok': 'Rok', 'Věková kategorie': 'Věková skupina'},
    category_orders={"Věková kategorie": ['0–19', '20–44', '45–64', '65–74', '75–84', '85+']},
    color_discrete_sequence=px.colors.sequential.Blues_r,
    template='plotly_white'
)

fig_flu_share.update_layout(barmode='relative', barnorm='percent')
fig_flu_share.show()

In [10]:

mapping_covid = {
    '0–19': ['Pozitivní ve věku 0-4 let', 'Pozitivní ve věku 5-9 let', 'Pozitivní ve věku 10-14 let', 'Pozitivní ve věku 15-19 let'],
    '20–44': ['Pozitivní ve věku 20-24 let', 'Pozitivní ve věku 25-29 let', 'Pozitivní ve věku 30-34 let', 'Pozitivní ve věku 35-39 let', 'Pozitivní ve věku 40-44 let'],
    '45–64': ['Pozitivní ve věku 45-49 let', 'Pozitivní ve věku 50-54 let', 'Pozitivní ve věku 55-59 let', 'Pozitivní ve věku 60-64 let'],
    '65–74': ['Pozitivní ve věku 65-69 let', 'Pozitivní ve věku 70-74 let'],
    '75–84': ['Pozitivní ve věku 75-79 let', 'Pozitivní ve věku 80-84 let'],
    '85+': ['Pozitivní ve věku 85+ let']
}

cfr_list = []
for skupina, sloupce in mapping_covid.items():
    nakazeni = df_covid_prep.groupby('rok')[sloupce].sum().sum(axis=1)
    umrti = df_covid_age_total[df_covid_age_total['vekova_skupina'] == skupina].set_index('rok')['pocet_umrti']
    
    cfr = (umrti / nakazeni) * 100
    temp_cfr = cfr.reset_index()
    temp_cfr.columns = ['rok', 'cfr_procenta']
    temp_cfr['vekova_skupina'] = skupina
    cfr_list.append(temp_cfr)

df_cfr_final = pd.concat(cfr_list)

fig_cfr = px.line(
    df_cfr_final, x='rok', y='cfr_procenta', color='vekova_skupina',
    title='COVID-19: Vývoj úmrtnosti nakažených dle věkových skupin',
    labels={'cfr_procenta': 'Úmrtnost v %', 'rok': 'Rok'},
    category_orders={"vekova_skupina": ['0–19', '20–44', '45–64', '65–74', '75–84', '85+']},
    template='plotly_white', markers=True
)
fig_cfr.show()

#### **Výpočet a časový vývoj smrtnosti (CFR) dle věkových skupin**

Zatímco předchozí analýzy sledovaly úmrtnost v rámci celé populace, tento blok kódu se zaměřuje na smrtnost (Case Fatality Rate – CFR), která vyjadřuje poměr mezi počtem zemřelých a počtem potvrzených případů nákazy. Tento ukazatel je klíčový pro pochopení věkově specifického rizika úmrtí při onemocnění COVID-19.

**Metodický postup výpočtu:**

    Párování nákaz a úmrtí: Kód využívá dříve definované věkové mapování k sčítání nově nakažených osob (pozitivních) v rámci stejných intervalů, ve kterých byla sledována úmrtnost.

    Výpočetní algoritmus: Pro každou věkovou skupinu a každý rok je vypočten podíl úmrtí k počtu nakažených a výsledek je normalizován na procenta. Tato operace probíhá iterativně napříč celým časovým obdobím 2020–2026.

    Logaritmické měřítko rizik: Spojnicový graf (px.line) s markery vizualizuje, jak se pravděpodobnost úmrtí v případě nákazy dramaticky liší mezi mladšími ročníky a seniorskou populací.

**Význam pro interpretaci výsledků:**
Tato vizualizace umožňuje v diplomové práci exaktně doložit dva zásadní fenomény:

    Věk jako dominantní rizikový faktor: Propastný rozdíl v hodnotách CFR mezi skupinou 85+ a mladšími ročníky potvrzuje biologickou selektivitu viru.

    Efekt medicínského pokroku a vakcinace: Pokles křivek v čase (zejména u rizikových skupin) dokumentuje zvyšující se imunitu populace a efektivitu léčebných protokolů, což vedlo k postupnému snižování „smrtnosti“ onemocnění navzdory pokračující cirkulaci viru.

In [11]:

# Klíč pro vek_kat: 
#   • Název parametru: vek_kat
#   • Datový typ: numeric 
#   • Popis parametru: kód 5leté věkové kategorie pacienta v době poskytnutí zdravotní služby vycházející z číselníku věkových skupin
#   • Možné hodnoty: kódy 5letých věkových skupin vycházející z číselníku věkových skupin; kódy odpovídají hodnotám věku 0–94 a 95+
mapping_chripka_jednotny = {
    '66000004': '0–19', '66005009': '0–19', '66010014': '0–19', '66015019': '0–19',
    '66020024': '20–44', '66025029': '20–44', '66030034': '20–44', '66035039': '20–44', '66040044': '20–44',
    '66045049': '45–64', '66050054': '45–64', '66055059': '45–64', '66060064': '45–64',
    '66065069': '65–74', '66070074': '65–74',
    '66075079': '75–84', '66080084': '75–84',
    '66085089': '85+', '66090094': '85+', '66095999': '85+' 
}

df_flu_final = df_chripka_umrti_hosp_kraje.copy()
df_flu_final['vek_kat'] = df_flu_final['vek_kat'].astype(str)
df_flu_final['Věková kategorie'] = df_flu_final['vek_kat'].map(mapping_chripka_jednotny)

df_flu_hfr = df_flu_final.groupby(['rok', 'Věková kategorie']).agg({
    'pocet_hosp': 'sum',
    'umrti': 'sum'
}).reset_index()

df_flu_hfr['hfr_procenta'] = (df_flu_hfr['umrti'] / df_flu_hfr['pocet_hosp']) * 100

fig_flu_hfr = px.line(
    df_flu_hfr, 
    x='rok', y='hfr_procenta', color='Věková kategorie',
    title='Chřipka: Vývoj úmrtnosti hospitalizovaných dle věkových skupin',
    labels={'hfr_procenta': 'Úmrtnosti v %', 'rok': 'Rok'},
    category_orders={"Věková kategorie": ['0–19', '20–44', '45–64', '65–74', '75–84', '85+']},
    markers=True,
    template='plotly_white',
    color_discrete_sequence=px.colors.sequential.Blues_r
)

fig_flu_hfr.update_yaxes(range=[0, 25])
fig_flu_hfr.show()

#### **Analýza preventivních opatření: Kvantifikace vakcinace v čase**

Tato část analýzy se zaměřuje na dynamiku očkovací kampaně proti COVID-19 a dlouhodobý vývoj vakcinace proti sezónní chřipce. Cílem je sjednotit metodiku vykazování „chráněných osob“, což je komplikováno odlišnými schématy u různých typů vakcín a sezónním charakterem sběru dat u chřipky.

**Klíčové metodické operace:**

    Algoritmus pro stanovení plné ochrany (COVID-19): Kód implementuje logickou funkci get_protected_persons, která rozlišuje mezi jednodávkovými schématy (vakcína Janssen) a vícedávkovými protokoly. Tím je zajištěno, že reportovaný počet osob skutečně odpovídá dokončenému základnímu očkování, nikoliv pouze počtu vyočkovaných ampulí.

    Transformace sezónních dat na kalendářní (Chřipka): Data o chřipce jsou primárně evidována v sezónách (např. 2023–2024). Kód provádí extrakci druhého roku sezóny, aby bylo možné data vizualizovat na stejné časové ose jako data o COVID-19.

    Vizualizace s duální osou: Grafy využívají knihovnu go.Figure k zobrazení absolutního objemu očkovaných osob (sloupce). Struktura grafu je připravena na zobrazení procentuální proočkovanosti na sekundární ose y, což umožňuje přímo sledovat nasycení populace vakcínou.

Význam pro interpretaci:
Zobrazené trendy umožňují v analytické části práce kriticky zhodnotit korelaci mezi masivním nástupem vakcinace (u COVID-19 v roce 2021) a následným vývojem smrtnosti (CFR). U chřipky pak graf odhaluje dlouhodobou stabilitu či volatilitu zájmu o prevenci v české populaci, což slouží jako referenční rámec pro hodnocení úspěšnosti pandemických opatření.

In [12]:

# COVID-19: Výpočet plně chráněných osob
df_covid_ockov_lidi = df_covid_ockov_kraje.copy()
df_covid_ockov_lidi['datum_dt'] = pd.to_datetime(df_covid_ockov_lidi['datum'], dayfirst=True)

# Rozlišení očkování podle potřebných dávek
def get_protected_persons(row):
    if 'Janssen' in str(row['vakcina']):
        return row['prvnich_davek']
    else:
        return row['druhych_davek']

df_covid_ockov_lidi['chranene_osoby'] = df_covid_ockov_lidi.apply(get_protected_persons, axis=1)

df_covid_vax_rocni = df_covid_ockov_lidi.assign(rok=df_covid_ockov_lidi['datum_dt'].dt.year).groupby('rok').agg({
    'chranene_osoby': 'sum'
}).reset_index()

# Chřipka: Převod ze sezón na roky
df_chripka_vax_rocni = df_chripka_ockov_vsichni_kraje.copy()
df_chripka_vax_rocni['rok'] = df_chripka_vax_rocni['sezona'].str.split('-').str[1].astype(int)
df_chripka_vax_rocni = df_chripka_vax_rocni.groupby('rok').agg({
    'pocet_vakcinovanych': 'sum',
    'proockovanost_procenta': 'mean'
}).reset_index()

colors_vax = {'objem': 'seagreen', 'procento': 'black'}

def create_vaccination_chart(df, title, count_col, perc_col=None):
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=df['rok'],
        y=df[count_col],
        name='Očkované osoby',
        marker_color=colors_vax['objem'],
        opacity=0.7,
        yaxis='y1'
    ))

    fig.update_layout(
        title=title,
        xaxis_title='Rok',
        yaxis=dict(title='Počet osob (kumulativně za rok)', side='left'),
        yaxis2=dict(title='Proočkovanost (%)', overlaying='y', side='right', range=[0, 100], showgrid=False),
        template='plotly_white',
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        hovermode="x unified"
    )
    return fig


# Graf pro COVID-19
fig_vax_covid = create_vaccination_chart(df_covid_vax_rocni, 'COVID-19: Roční přehled plně očkovaných osob', 'chranene_osoby')
fig_vax_covid.show()

# Graf pro Chřipku
fig_vax_chripka = create_vaccination_chart(df_chripka_vax_rocni, 'Chřipka: Roční přehled očkovaných osob (Historie)', 'pocet_vakcinovanych', 'proockovanost_procenta')
fig_vax_chripka.show()

#### **Korelační analýza mortality a proočkovanosti v čase**

Tato fáze analýzy představuje syntézu epidemiologických a preventivních dat. Cílem je identifikovat statistickou souvislost mezi nárůstem proočkovanosti populace a vývojem měsíční úmrtnosti na COVID-19. Výpočet přechází od ročních sumářů k detailnější měsíční granulitě, která lépe zachycuje dynamiku jednotlivých vln pandemie.

**Metodický postup a výpočetní operace:**

    Datová fúze (Merge): Kód provádí sjednocení denních záznamů o úmrtí a očkování na společnou časovou osu datum_dt. Chybějící hodnoty (např. dny bez hlášeného očkování) jsou ošetřeny metodou fillna(0), aby nedošlo k deformaci výsledné sumy.

    Výpočet kumulativní ochrany: Pomocí funkce cumsum() je modelován postupný nárůst chráněných osob v populaci. Výsledek je normalizován na procenta vzhledem k celkové populaci ČR (uvažováno 10,7 mil. obyvatel).

    Statistická verifikace (Pearsonova korelace): Přímo v rámci vizualizace je vypočten koeficient r a p-hodnota. Tento výpočet kvantifikuje sílu vztahu: záporná hodnota r zde indikuje, že s rostoucí proočkovaností docházelo k poklesu mortality (při zohlednění dalších faktorů).

    Vizualizace na duálních osách: Graf kombinuje měsíční úmrtnost (sloupce na primární ose) s progresí proočkovanosti (spojnice na sekundární ose).

Analytický význam:Tento výstup umožňuje v textu práce exaktně diskutovat efektivitu preventivních opatření. Zatímco první vlny pandemie vykazovaly vysokou mortalitu při nulové ochraně, pozdější fáze (v roce 2022 a dále) ukazují stabilitu systému i při pokračující cirkulaci viru, což lze dát do přímé souvislosti s hladinou kolektivní imunity zobrazenou na sekundární ose.

In [13]:
df_vax_prep = df_covid_ockov_kraje.copy()
df_vax_prep['datum_dt'] = pd.to_datetime(df_vax_prep['datum'], dayfirst=True, errors='coerce')

df_vax_prep['chranene_osoby'] = df_vax_prep.apply(get_protected_persons, axis=1)

df_vax_daily = df_vax_prep.groupby('datum_dt')['chranene_osoby'].sum().reset_index()

df_deaths_prep = df_covid_umrti_hosp_cr.copy()
df_deaths_prep['datum_dt'] = pd.to_datetime(df_deaths_prep['Datum'], errors='coerce')

df_combined = pd.merge(df_deaths_prep, df_vax_daily, on='datum_dt', how='left')
df_combined['chranene_osoby'] = df_combined['chranene_osoby'].fillna(0)
df_combined['Počet zemřelých'] = df_combined['Počet zemřelých'].fillna(0)


df_combined['mesic'] = df_combined['datum_dt'].dt.to_period('M').astype(str)

covid_monthly = df_combined.groupby('mesic').agg({
    'Počet zemřelých': 'sum',
    'chranene_osoby': 'sum'
}).reset_index()


covid_monthly['vax_kumulativni'] = covid_monthly['chranene_osoby'].cumsum()

covid_monthly['vax_perc'] = (covid_monthly['vax_kumulativni'] / 10700000) * 100

fig_covid_time = make_subplots(specs=[[{"secondary_y": True}]])

fig_covid_time.add_trace(
    go.Bar(x=covid_monthly['mesic'], y=covid_monthly['Počet zemřelých'], 
           name="Počet úmrtí", marker_color='firebrick', opacity=0.7),
    secondary_y=False,
)

fig_covid_time.add_trace(
    go.Scatter(x=covid_monthly['mesic'], y=covid_monthly['vax_perc'], 
               name="Proočkovanost (%)", line=dict(color='seagreen', width=4)),
    secondary_y=True,
)

fig_covid_time.update_layout(
    title='COVID-19: Časový vývoj úmrtnosti v kontextu proočkovanosti',
    xaxis_title='Měsíc',
    template='plotly_white',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    hovermode="x unified"
)

fig_covid_time.update_yaxes(title_text="Počet úmrtí (měsíčně)", secondary_y=False)
fig_covid_time.update_yaxes(title_text="Plně očkovaná populace (%)", secondary_y=True, range=[0, 100])

# 1. PŘÍPRAVA DAT PRO ANALÝZU
clean_covid = covid_monthly[['Počet zemřelých', 'vax_perc']].dropna()

# 2. VÝPOČET PEARSONOVY KORELACE
r_covid, p_covid = stats.pearsonr(clean_covid['vax_perc'], clean_covid['Počet zemřelých'])

# 3. ZOBRAZENÍ STATISTIK V GRAFU
fig_covid_time.add_annotation(
    xref="paper", yref="paper",
    x=0.02, y=0.95,
    text=f"r = {r_covid:.2f}<br>p-hodnota {'< 0.001' if p_covid < 0.001 else f'= {p_covid:.3f}'}",
    showarrow=False,
    font=dict(size=14, color="black"),
    bgcolor="white",
    bordercolor="black",
    borderwidth=1,
    align="left"
)

fig_covid_time.show()

#### **Statistická validace: Regresní analýza vztahu proočkovanosti a mortality**

Pro exaktní potvrzení vztahu mezi mírou proočkovanosti populace a měsíčním počtem úmrtí na COVID-19 je v tomto bloku využita regresní analýza. Bodový graf vizualizuje rozptyl jednotlivých měsíčních pozorování, přičemž každé z nich reprezentuje konkrétní stav proočkovanosti a odpovídající počet obětí.

**Klíčové analytické prvky:**

    Lineární regrese (Metoda nejmenších čtverců - OLS): Parametr trendline="ols" automaticky prokládá daty regresní přímku. Sklon této přímky vizuálně potvrzuje směr závislosti – klesající tendence přímky jasně dokládá, že s rostoucí hladinou proočkovanosti docházelo ke snižování absolutní měsíční úmrtnosti.

    Interpretace rozptylu: Pozice bodů mimo regresní čáru (outliers) indikuje období, kdy do hry vstupovaly jiné faktory, jako byla agresivita konkrétních variant viru (např. vlna Delta) nebo sezónní vlivy, které mohly dočasně zvýšit mortalitu i při vyšší proočkovanosti.

    Korelační koeficient (r): Hodnota r uvedená v nadpisu grafu poskytuje čtenáři okamžitou informaci o těsnosti tohoto vztahu. Záporná hodnota (v tomto případě pravděpodobně silně negativní) je klíčovým argumentem pro závěrečnou diskusi práce o efektivitě veřejno-zdravotních opatření.

Význam pro diplomovou práci:Tento graf je syntetickým výstupem celé praktické části. Umožňuje shrnout tisíce řádků epidemiologických dat do jediného srozumitelného argumentu, který statisticky obhajuje přínos vakcinace v boji proti pandemické úmrtnosti v České republice.

In [14]:
# COVID-19: Bodový graf s regresní čárou
fig_corr_covid = px.scatter(
    covid_monthly, 
    x='vax_perc', 
    y='Počet zemřelých',
    trendline="ols", # Přidá automaticky regresní přímku
    title=f'Korelace: Proočkovanost vs. Úmrtnost (COVID-19) | r = {r_covid:.2f}',
    labels={'vax_perc': 'Proočkovanost (%)', 'Počet zemřelých': 'Počet úmrtí (měsíčně)'},
    template='plotly_white',
    color_discrete_sequence=['firebrick']
)

fig_corr_covid.update_layout(hovermode="closest")
fig_corr_covid.show()

#### **Korelační analýza mortality a proočkovanosti u sezónní chřipky**

Tento blok kódu provádí retrospektivní analýzu vztahu mezi úrovní vakcinace a celkovou úmrtností na chřipku v České republice za období 2011–2023. Cílem je vytvořit metodický protipól k analýze COVID-19 a zjistit, zda lze i u běžné sezónní chřipky identifikovat statisticky významnou závislost mezi prevencí a následky onemocnění.

**Klíčové metodické kroky:**

    Sjednocení sezónních a kalendářních dat: Vzhledem k tomu, že chřipkové sezóny překrývají dva kalendářní roky, kód provádí extrakci koncového roku sezóny pro korektní spárování s ročními statistikami úmrtí.

    Agregace mortality: Data jsou filtrována pro kategorii „Celá populace“, čímž získáme absolutní roční počty obětí, které jsou následně očištěny od regionálních vlivů.

    Statistická komparace: Stejně jako u COVID-19 je vypočten Pearsonův korelační koeficient (r). U chřipky je tento výpočet obzvláště zajímavý z hlediska interpretace p-hodnoty, která může odhalit, zda je nízká hladina proočkovanosti v ČR (zobrazená na ose y v rozsahu 0–15 %) dostatečná k prokazatelnému ovlivnění celkové mortality.

    Vizualizace trendů: Graf využívá duální osu y pro zobrazení ročních úmrtí (sloupce) a průměrné proočkovanosti (spojnice), což umožňuje vizuálně identifikovat roky s extrémní epidemickou zátěží.

Analytický význam pro závěry práce:Tento výstup je zásadní pro komparativní diskusi. Umožňuje autorovi srovnat stabilitu proočkovanosti u chřipky s dynamickým (a často jednorázovým) nárůstem u COVID-19. Výsledná korelace u chřipky slouží jako referenční bod: pokud je r blízké nule, potvrzuje to hypotézu, že při velmi nízké hladině proočkovanosti populace (pod 10 %) je vliv vakcinace na celonárodní mortalitu obtížně statisticky prokazatelný ve srovnání s vnějšími vlivy (např. agresivita konkrétního kmene viru).

In [15]:
df_flu_deaths_sum = df_chripka_umrti_abs_kraje[df_chripka_umrti_abs_kraje['Věková kategorie'] == 'Celá populace'].copy()
roky_sezony = [str(r) for r in range(2011, 2024)]
df_flu_deaths_final = df_flu_deaths_sum[roky_sezony].sum().reset_index()
df_flu_deaths_final.columns = ['rok', 'pocet_umrti']

df_flu_vax_agg = df_chripka_ockov_vsichni_kraje.copy()
df_flu_vax_agg['rok'] = df_flu_vax_agg['sezona'].str.split('-').str[1]
df_flu_vax_final = df_flu_vax_agg.groupby('rok').agg({
    'proockovanost_procenta': 'mean'
}).reset_index()

df_flu_combined = pd.merge(df_flu_deaths_final, df_flu_vax_final, on='rok')

fig_flu_time = make_subplots(specs=[[{"secondary_y": True}]])

fig_flu_time.add_trace(
    go.Bar(x=df_flu_combined['rok'], y=df_flu_combined['pocet_umrti'], 
           name="Počet úmrtí", marker_color='firebrick', opacity=0.7),
    secondary_y=False,
)

fig_flu_time.add_trace(
    go.Scatter(x=df_flu_combined['rok'], y=df_flu_combined['proockovanost_procenta'], 
               name="Proočkovanost (%)", line=dict(color='seagreen', width=3)),
    secondary_y=True,
)

fig_flu_time.update_layout(
    title='Chřipka: Srovnání mortality a proočkovanosti v sezónách',
    xaxis_title='Sezóna (koncový rok)',
    template='plotly_white',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig_flu_time.update_yaxes(title_text="Počet úmrtí (ročně)", secondary_y=False)
fig_flu_time.update_yaxes(title_text="Proočkovanost (%)", secondary_y=True, range=[0, 15])

clean_flu = df_flu_combined[['pocet_umrti', 'proockovanost_procenta']].dropna()

r_flu, p_flu = stats.pearsonr(clean_flu['proockovanost_procenta'], clean_flu['pocet_umrti'])

fig_flu_time.add_annotation(
    xref="paper", yref="paper",
    x=0.02, y=0.95,
    text=f"r = {r_flu:.2f}<br>p-hodnota {'< 0.001' if p_flu < 0.001 else f'= {p_flu:.3f}'}",
    showarrow=False,
    font=dict(size=14, color="black"),
    bgcolor="white",
    bordercolor="black",
    borderwidth=1,
    align="left"
)

fig_flu_time.show()

#### **Statistická validace: Regresní analýza vztahu proočkovanosti a mortality u chřipky**

Závěrečným krokem kvantitativní analýzy je zobrazení závislosti mezi mírou proočkovanosti a ročním počtem úmrtí na chřipku pomocí bodového grafu s lineární regresí. Tento výstup slouží k přímému srovnání s regresním modelem COVID-19 a umožňuje posoudit efektivitu vakcinace v podmínkách nízké, ale stabilní proočkovanosti populace ČR.

**Analytické charakteristiky:**

    Regresní přímka (OLS): Parametr trendline="ols" prokládá body přímku, která vizualizuje celkový trend. Vzhledem k historicky nízké a málo variabilní proočkovanosti proti chřipce (pohybuje se většinou v úzkém pásmu 5–10 %) je sklon této přímky klíčovým indikátorem pro diskusi o reálném dopadu prevence na celonárodní úrovni.

    Interpretace korelace (r): Hodnota korelačního koeficientu v nadpisu grafu exaktně vyjadřuje, zda existuje prokazatelná vazba mezi objemem očkovaných a počtem obětí. Pokud je r blízké nule, potvrzuje to, že u chřipky v ČR hrají dominantní roli jiné faktory (agresivita variant viru, klimatické podmínky), které přebíjejí vliv současné úrovně vakcinace.

    Komparativní potenciál: Bodový graf pro chřipku (v zelených odstínech) tvoří přímý metodický protipól ke grafu pro COVID-19 (v červených odstínech). Toto srovnání je v diplomové práci zásadní pro vyhodnocení rozdílů mezi pandemickým a endemickým průběhem onemocnění.

In [16]:
# CHŘIPKA: Bodový graf s regresní čárou
fig_corr_flu = px.scatter(
    df_flu_combined, 
    x='proockovanost_procenta', 
    y='pocet_umrti',
    trendline="ols",
    title=f'Korelace: Proočkovanost vs. Úmrtnost (Chřipka) | r = {r_flu:.2f}',
    labels={'proockovanost_procenta': 'Proočkovanost (%)', 'pocet_umrti': 'Počet úmrtí (ročně)'},
    template='plotly_white',
    color_discrete_sequence=['seagreen']
)

fig_corr_flu.update_layout(hovermode="closest")
fig_corr_flu.show()

#### **Regionální srovnání: Normalizace epidemiologických dat na populaci krajů**

Pro objektivní srovnání situace v jednotlivých krajích ČR je nezbytné převést absolutní počty (úmrtí, hospitalizace, očkování) na relativní ukazatele vztažené k počtu obyvatel. Tento blok kódu provádí komplexní integraci dat z roku 2021, který byl z hlediska pandemické dynamiky i vakcinační kampaně v České republice nejvýznamnější.

**Provedené analytické operace:**

    Demografická normalizace: Pomocí externího slovníku populace_kraje_map jsou vypočteny relativní metriky (procentuální proočkovanost, úmrtnost a hospitalizace na obyvatele). To eliminuje zkreslení způsobené rozdílnou velikostí krajů (např. srovnání Prahy s Karlovarským krajem).

    Agregace diagnostického tlaku: Výpočet počtu testů na 100 000 obyvatel (tests_per_100k) umožňuje posoudit, zda v některých krajích nebyla úmrtnost podhodnocena kvůli nižší intenzitě testování.

    Integrace vakcinačních dat: Kód aplikuje logiku „plně chráněných osob“ (Janssen vs. ostatní vakcíny) na úrovni krajských NUTS kódů.

    Seřazení dle úspěšnosti (Ranking): Výsledný DataFrame df_reg je seřazen podle proočkovanosti, což okamžitě odhaluje regionální lídry a zaostávající kraje v preventivní kampani.

Význam pro diplomovou práci:
Tato tabulka tvoří datový základ pro geografickou analýzu pandemie. Umožňuje v textu práce diskutovat, zda existuje „regionální nůžky“ – tedy zda například strukturálně znevýhodněné kraje vykazovaly jiný vztah mezi proočkovaností a úmrtností než kraje s vyšší dostupností zdravotní péče a diagnostiky.

In [17]:
populace_kraje_map = {
    'CZ010': 1335084, 'CZ020': 1397997, 'CZ031': 643551, 'CZ032': 591041,
    'CZ041': 293311,  'CZ042': 817004,  'CZ051': 442476, 'CZ052': 550803,
    'CZ053': 522856,  'CZ063': 508852,  'CZ064': 1195327, 'CZ071': 630522,
    'CZ072': 580119,  'CZ080': 1192834
}

df_cov_h = df_covid_umrti_hosp_kraje.copy()
df_cov_h['Datum'] = pd.to_datetime(df_cov_h['Datum'])
df_2021 = df_cov_h[df_cov_h['Datum'].dt.year == 2021]

df_metrics = df_2021.groupby('Kraj_ID').agg({
    'Počet zemřelých': 'sum',
    'Počet nově hospitalizovaných celkem': 'sum',
    'počet testů CELKEM': 'sum'
}).reset_index()

df_vax_k = df_covid_ockov_kraje.copy()
df_vax_k['chraneni'] = df_vax_k.apply(lambda x: x['prvnich_davek'] if 'Janssen' in str(x['vakcina']) else x['druhych_davek'], axis=1)
df_vax_agg = df_vax_k.groupby('kraj_nuts_kod')['chraneni'].sum().reset_index()

df_reg = pd.merge(df_metrics, df_vax_agg, left_on='Kraj_ID', right_on='kraj_nuts_kod')
df_reg['populace'] = df_reg['Kraj_ID'].map(populace_kraje_map)
df_reg['kraj_nazev'] = df_reg['Kraj_ID'].map(kod_kraje_to_nazev)

df_reg['vax_perc'] = (df_reg['chraneni'] / df_reg['populace']) * 100
df_reg['death_perc'] = (df_reg['Počet zemřelých'] / df_reg['populace']) * 100
df_reg['hosp_perc'] = (df_reg['Počet nově hospitalizovaných celkem'] / df_reg['populace']) * 100
df_reg['tests_per_100k'] = (df_reg['počet testů CELKEM'] / df_reg['populace']) * 100000

df_reg = df_reg.sort_values(by='vax_perc', ascending=False)

#### **Analýza regionální divergence: Vztah proočkovanosti a úmrtnosti (Efekt nůžek)**

Tento oddíl se zaměřuje na ověření hypotézy, zda vyšší míra proočkovanosti v konkrétních krajích vedla k měřitelnému snížení relativní úmrtnosti na COVID-19. Vizualizace kombinuje barový graf (míra prevence) se spojnicovým grafem (míra následků) pro všech 14 krajů ČR.

**Klíčové prvky analýzy:**

    Duální osa y: Umožňuje přímé srovnání dvou řádově odlišných veličin – procenta proočkované populace (osa vlevo) a procenta úmrtí vztaženého k celkovému počtu obyvatel kraje (osa vpravo).

    Vizuální identifikace „nůžek“: Cílem grafu je ukázat, zda kraje s nejvyššími sloupci (vysoká proočkovanost) vykazují zároveň nejnižší body na červené křivce (nízká úmrtnost). Případná divergence (rozevírání nůžek) je jasným indikátorem úspěšnosti očkovací kampaně v daném regionu.

    Pearsonova korelace (r): Výpočet korelačního koeficientu na úrovni krajů poskytuje exaktní odpověď na otázku, jak silná byla vazba mezi ochranou a úmrtností v rámci celého území ČR. Výsledná p-hodnota určuje statistickou významnost tohoto vztahu (zda rozdíly nejsou dány pouze náhodou).

In [18]:
fig_nuzky = go.Figure()
fig_nuzky.add_trace(go.Bar(x=df_reg['kraj_nazev'], y=df_reg['vax_perc'], name='Proočkovanost (%)', marker_color='seagreen', opacity=0.8, yaxis='y1'))
fig_nuzky.add_trace(go.Scatter(x=df_reg['kraj_nazev'], y=df_reg['death_perc'], name='Úmrtnost (%)', marker_color='firebrick', mode='lines+markers', yaxis='y2'))

fig_nuzky.update_layout(
    title='Srovnání ochrany a následků (COVID-19, 2021)',
    yaxis=dict(title='Proočkovanost (%)', side='left', range=[0, 100]),
    yaxis2=dict(title='Úmrtnost v % populace', side='right', overlaying='y', range=[0, 0.5], showgrid=False),
    template='plotly_white', legend=dict(x=0.01, y=0.99)
)
fig_nuzky.show()

# Korelace mezi proočkovaností a úmrtností v krajích za rok 2021
r_vax_death, p_value = stats.pearsonr(df_reg['vax_perc'], df_reg['death_perc'])

print(f"Korelační koeficient (r): {r_vax_death:.4f}")
print(f"P-hodnota: {p_value:.4f}")

Korelační koeficient (r): -0.3000
P-hodnota: 0.2974


#### **Multikriteriální analýza regionální zátěže (Heatmapa zranitelnosti)**

Pro finální srovnání čtrnácti krajů ČR byla zvolena forma normalizované heatmapy. Tato metoda umožňuje vizualizovat nesourodé metriky (procenta očkovaných vs. počty testů na 100 tis. obyvatel) na jednotné bezrozměrné škále od 0 do 1, kde 0 představuje nejnižší a 1 nejvyšší hodnotu v rámci daného ukazatele mezi kraji.

**Metodický přínos a zpracování:**

    Min-Max Normalizace: Transformace dat zajišťuje, že extrémní hodnoty v jedné metrice (např. vysoký počet testů) nezastíní jemnější rozdíly v jiných (např. úmrtnost).

    Barevná sémantika (RdYlGn_r): Byla zvolena reverzní škála (červená–žlutá–zelená), která intuitivně signalizuje rizikovost. Červená barva indikuje nepříznivý stav (vysoká úmrtnost, nízká proočkovanost), zatímco zelená značí pozitivní epidemiologickou bilanci.

    Identifikace anomálií: Mapa umožňuje okamžitě detekovat kraje, které vykazují vysokou intenzitu testování, ale zároveň nízkou úmrtnost (úspěšná diagnostika), nebo naopak regiony s vysokou hospitalizací navzdory průměrné proočkovanosti.

In [19]:
# Heatmapa zátěže
heatmap_df = df_reg[['kraj_nazev', 'vax_perc', 'hosp_perc', 'death_perc', 'tests_per_100k']].set_index('kraj_nazev')
heatmap_norm = (heatmap_df - heatmap_df.min()) / (heatmap_df.max() - heatmap_df.min())

fig_heatmap = px.imshow(
    heatmap_norm.T,
    labels=dict(x="Kraj", y="Metrika", color="Relativní zátěž"),
    y=['Proočkovanost', 'Hospitalizace', 'Úmrtnost', 'Intenzita testování'],
    color_continuous_scale='RdYlGn_r',
    title='Mapa zranitelnosti krajů (Relativní srovnání zátěže, 2021)'
)
fig_heatmap.show()

#### **Souhrnný přehled regionálních epidemiologických indikátorů (2021)**

Závěrečný krok datové části představuje konsolidaci všech klíčových metrik do podoby strukturovaného reportu. Tabulka final_table shrnuje výsledky ročního úsilí v oblasti prevence (vakcinace), diagnostiky (testování) a následků pandemie (hospitalizace a úmrtnost) pro všech 14 krajů České republiky.

In [20]:
final_table = df_reg[['kraj_nazev', 'vax_perc', 'death_perc', 'hosp_perc', 'tests_per_100k']].round(3)
final_table.columns = ['Kraj', 'Proočkovanost (%)', 'Úmrtnost (%)', 'Hospitalizace (%)', 'Testy na 100 tis.']
display(final_table)

,Kraj,Proočkovanost (%),Úmrtnost (%),Hospitalizace (%),Testy na 100 tis.
0,Hlavní město Praha,96.840,0.160,0.869,488687.154
10,Jihomoravský kraj,65.015,0.238,1.226,363969.357
2,Jihočeský kraj,63.642,0.243,1.241,334311.344
7,Královéhradecký kraj,63.636,0.247,1.136,387469.567
3,Plzeňský kraj,63.464,0.272,1.154,388537.174
4,Karlovarský kraj,61.566,0.387,1.365,304891.736
9,Kraj Vysočina,61.211,0.182,1.147,339180.744
6,Liberecký kraj,60.961,0.221,1.064,352331.878
8,Pardubický kraj,59.721,0.240,1.252,386387.648
5,Ústecký kraj,59.584,0.260,1.339,365608.614


#### **Celková bilance: Komparace kumulativních dopadů COVID-19 a sezónní chřipky**

Závěrečný analytický výstup práce přináší přímé srovnání celkové mortality a nemocniční zátěže obou sledovaných onemocnění. Aby byla zachována maximální objektivita, jsou data pro chřipku sumována za celé dostupné období tří desetiletí (1994–2023), zatímco data pro COVID-19 pokrývají období od počátku pandemie (2020–2025).

**Klíčové analytické aspekty:**

    Asymetrické časové srovnání: Kód počítá kumulativní sumy úmrtí a hospitalizací pro oba patogeny. Výsledný nepoměr (ratio) je nejsilnějším argumentem pro pochopení unikátnosti pandemické situace.

    Vizualizace absolutních dopadů: Sloupcový graf (go.Bar) s textovými popisky nad sloupci umožňuje okamžitý odečet rozdílů v řádech. Použití barev (červená pro COVID-19, modrá pro chřipku) udržuje jednotný vizuální styl celého dokumentu.

    Kvantifikace závažnosti: Výpočet poměrů (death_ratio a hosp_ratio) dává suchým číslům srozumitelný rozměr – např. kolikrát více úmrtí způsobil COVID-19 za zlomek času ve srovnání s dlouhodobým historickým průměrem chřipky.

Význam pro závěrečnou diskusi diplomové práce:
Tento výstup slouží jako „finální verdikt“. Empiricky dokazuje, že COVID-19 nebyl pouhou „silnější chřipkou“, ale jevem, který v řádu několika málo let překonal kumulativní dopady sezónní chřipky za posledních 30 let. Tato data jsou nezbytná pro obhajobu nutnosti přijatých mimořádných opatření a pro pochopení tlaku, kterému musel český zdravotní systém v letech 2020–2022 čelit.

In [47]:
total_deaths_covid = df_covid_umrti_hosp_cr['Počet zemřelých'].sum()
total_hosp_covid = df_covid_umrti_hosp_cr['Počet nově hospitalizovaných celkem'].sum()

total_deaths_flu = len(df_chripka_umrti_kraje)

total_hosp_flu = df_chripka_umrti_hosp_kraje['pocet_hosp'].sum()

categories = ['Celkový počet úmrtí', 'Celkový počet hospitalizací']

fig_final_compare = go.Figure()

fig_final_compare.add_trace(go.Bar(
    x=categories,
    y=[total_deaths_covid, total_hosp_covid],
    name='COVID-19 (2020–2025)',
    marker_color='firebrick',
    text=[f"{total_deaths_covid:,.0f}", f"{total_hosp_covid:,.0f}"],
    textposition='outside'
))

fig_final_compare.add_trace(go.Bar(
    x=categories,
    y=[total_deaths_flu, total_hosp_flu],
    name='Chřipka (1994–2024)',
    marker_color='royalblue',
    text=[f"{total_deaths_flu:,.0f}", f"{total_hosp_flu:,.0f}"],
    textposition='outside'
))

fig_final_compare.update_layout(
    title='Finální srovnání COVID-19 vs. Sezónní chřipka v ČR',
    yaxis_title='Absolutní počet osob',
    barmode='group',
    template='plotly_white',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(t=100) 
)

fig_final_compare.show()

death_ratio = total_deaths_covid / total_deaths_flu
hosp_ratio = total_hosp_covid / total_hosp_flu

print(f"--- ZÁVĚREČNÁ BILANCE ---")
print(f"COVID-19 způsobil za 5 roky {death_ratio:.1f}x více úmrtí než chřipka za 31 let.")
print(f"COVID-19 vyžadoval za 5 roky {hosp_ratio:.1f}x více hospitalizací než chřipka za 31 let.")

--- ZÁVĚREČNÁ BILANCE ---
COVID-19 způsobil za 5 roky 13.4x více úmrtí než chřipka za 31 let.
COVID-19 vyžadoval za 5 roky 4.4x více hospitalizací než chřipka za 31 let.


#### **Pomocné ukázky**

Tento závěrečný blok slouží jako metodická ukázka rozdílného vykazování úmrtí na chřipku v rámci různých informačních systémů ČR (absolutní statistiky vs. nemocniční hlášení), což odhaluje významný podíl úmrtí nastalých mimo lůžkovou péči a zdůrazňuje potřebu kritického přístupu při srovnávání těchto dat s detailněji sledovaným COVID-19.

In [51]:
roky_srovnani = [str(r) for r in range(1994, 2024)]

total_deaths_abs = df_chripka_umrti_abs_kraje.loc[
    (df_chripka_umrti_abs_kraje['Kraj'] == 'ČR') & 
    (df_chripka_umrti_abs_kraje['Věková kategorie'] == 'Celá populace'), 
    roky_srovnani
].sum(axis=1).values[0]

df_umrti_detail = df_chripka_umrti_kraje.copy()
if 'datum_umrti' in df_umrti_detail.columns:
    df_umrti_detail['rok'] = pd.to_datetime(df_umrti_detail['datum_umrti'], dayfirst=True).dt.year

total_deaths_detail = len(df_umrti_detail[
    (df_umrti_detail['rok'] >= 1994) & (df_umrti_detail['rok'] <= 2023)
])

df_flu_hosp_sub = df_chripka_umrti_hosp_kraje[
    (df_chripka_umrti_hosp_kraje['rok'] >= 1994) & 
    (df_chripka_umrti_hosp_kraje['rok'] <= 2023)
]
total_deaths_hosp = df_flu_hosp_sub['umrti'].sum()

# --- VÝPIS VÝSLEDKŮ ---
print(f"--- SROVNÁNÍ ÚMRTNOSTI NA CHŘIPKU v ČR (2015–2023) ---")
print(f"1. Soubor: {total_deaths_abs:.0f}")
print(f"2. Soubor: {total_deaths_detail}")
print(f"3. Soubor: {total_deaths_hosp:.0f}")
print("-" * 50)
print(f"Rozdíl mezi celkovou a nemocniční úmrtností: {total_deaths_abs - total_deaths_hosp:.0f} osob.")

--- SROVNÁNÍ ÚMRTNOSTI NA CHŘIPKU v ČR (2015–2023) ---
1. Soubor: 2932
2. Soubor: 2932
3. Soubor: 1052
--------------------------------------------------
Rozdíl mezi celkovou a nemocniční úmrtností: 1880 osob.


#### **Export dat pro Webovou aplikaci**

In [24]:
df_covid_umrti_hosp_kraje.to_csv("DataApp/covid_hosp_umrti_kraje.csv", index=False)
df_covid_ockov_kraje.to_csv("DataApp/covid_ockov_kraje.csv", index=False)
df_chripka_umrti_hosp_kraje.to_csv("DataApp/flu_hosp_umrti_kraje.csv", index=False)
df_chripka_umrti_kraje.to_csv("DataApp/flu_umrti_detail.csv", index=False)
df_chripka_umrti_abs_kraje.to_csv("DataApp/flu_umrti_abs.csv", index=False)
df_chripka_umrti_norm_kraje.to_csv("DataApp/flu_umrti_norm.csv", index=False)
df_chripka_umrti_perc_kraje.to_csv("DataApp/flu_umrti_perc.csv", index=False)
df_chripka_ockov_65plus_kraje.to_csv("DataApp/flu_ockov_65.csv", index=False)
df_chripka_ockov_vsichni_kraje.to_csv("DataApp/flu_ockov_vse.csv", index=False)
df_covid_umrti_hosp_cr.to_csv("DataApp/covid_cr_final.csv", index=False)

print("Všech 10 datasetů bylo úspěšně uloženo pro webovou aplikaci.")

Všech 10 datasetů bylo úspěšně uloženo pro webovou aplikaci.
